<a href="https://colab.research.google.com/github/chamarairesh1982/LearnPython/blob/main/week_10_Clustering_Practical_Wine_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clustering Algorithms and Evaluation using Wine Dataset

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, BisectingKMeans
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

## Load Dataset
Dataset Information: https://www.geeksforgeeks.org/machine-learning/wine-dataset/

In [ ]:
wine=load_wine()
X=pd.DataFrame(wine.data,columns=wine.feature_names)
X.head()

## Scale Features
Important: Distance-based clustering requires scaling.

In [ ]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

## PCA for Visualization
Reduce all the original features in the dataset to 2 principal components.

In [ ]:
pca=PCA(n_components=2)
X_pca=pca.fit_transform(X_scaled)
plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0],X_pca[:,1])
plt.title("Wine Dataset PCA Projection")
plt.show()

# K-Means
Hyperparameter: n_clusters

In [ ]:
wcss=[]
for k in range(1,11):
    model=KMeans(n_clusters=k,random_state=42)
    model.fit(X_scaled)
    wcss.append(model.inertia_)
plt.plot(range(1,11),wcss,marker="o")
plt.title("Elbow Method")
plt.xlabel("K")
plt.ylabel("WCSS")
plt.show()

In [ ]:
kmeans=KMeans(n_clusters=3,random_state=42)
kmeans_labels=kmeans.fit_predict(X_scaled)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca[:,0],y=X_pca[:,1],hue=kmeans_labels,palette="Set1")
plt.title("K-Means Clusters")
plt.show()

# Agglomerative Hierarchical Clustering
Hyperparameters: n_clusters, linkage

In [ ]:
plt.figure(figsize=(10,5))
dendrogram(linkage(X_scaled,method="ward"))
plt.title("Dendrogram")
plt.show()

In [ ]:
agg=AgglomerativeClustering(n_clusters=3,linkage="ward")
agg_labels=agg.fit_predict(X_scaled)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca[:,0],y=X_pca[:,1],hue=agg_labels,palette="Set2")
plt.title("Agglomerative Clusters")
plt.show()

# Divisive Hierarchical Clustering
Using BisectingKMeans approximation. Hyperparameter: n_clusters

In [ ]:
divisive=BisectingKMeans(n_clusters=3,random_state=42)
div_labels=divisive.fit_predict(X_scaled)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca[:,0],y=X_pca[:,1],hue=div_labels,palette="tab10")
plt.title("Divisive Hierarchical Clusters")
plt.show()

# DBSCAN
Hyperparameters: eps, min_samples. Noise points get label -1.

In [ ]:
dbscan=DBSCAN(eps=3.0,min_samples=5)
db_labels=dbscan.fit_predict(X_scaled)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca[:,0],y=X_pca[:,1],hue=db_labels,palette="tab10", s=80)
plt.title("DBSCAN Clusters and Noise")
plt.show()
print("Unique cluster labels:",set(db_labels))

## Evaluation Metrics

In [ ]:
def evaluate(X,labels):
    if len(set(labels))<2:
        return [None,None,None]
    return [silhouette_score(X,labels),calinski_harabasz_score(X,labels),davies_bouldin_score(X,labels)]

k_s,k_c,k_d=evaluate(X_scaled,kmeans_labels)
a_s,a_c,a_d=evaluate(X_scaled,agg_labels)
d_s,d_c,d_d=evaluate(X_scaled,div_labels)
mask=db_labels!=-1
b_s,b_c,b_d=evaluate(X_scaled[mask],db_labels[mask])

In [ ]:
results=pd.DataFrame({
"Algorithm":["K-Means","Agglomerative","Divisive","DBSCAN"],
"Silhouette":[k_s,a_s,d_s,b_s],
"Calinski-Harabasz":[k_c,a_c,d_c,b_c],
"Davies-Bouldin":[k_d,a_d,d_d,b_d]})
results